# Ondas electromagnéticas
## Método FDTD

- Dos campos vectoriales acoplados: eléctrico y magnético.
- Ortogonales entre sí y a la dirección de propagación.

Para una onda propagándose en la dirección z, con el campo eléctrico en x y el campo magnético en y:

$$\mathbf{E} = E_x(z,t)\hat{x}$$

$$\mathbf{H} = H_y(z,t)\hat{y}$$

$$\mathbf{S} = \mathbf{E} \times \mathbf{H}$$

## Ecuaciones de Maxwell en el vacío

$$\begin{align}
\nabla \cdot \mathbf{E} &= 0 \\
\nabla \cdot \mathbf{H} &= 0 \\
\nabla \times \mathbf{E} &= -\mu_0 \frac{\partial \mathbf{H}}{\partial t} \\
\nabla \times \mathbf{H} &= \epsilon_0 \frac{\partial \mathbf{E}}{\partial t}
\end{align}$$

Si sustituimos las ecuaciones de campo eléctrico y magnético en las últimas dos ecuaciones de Maxwell, obtenemos:

$$\begin{align}
\frac{\partial E_x}{\partial z} &= -\mu_0 \frac{\partial H_y}{\partial t} \\
\frac{\partial H_y}{\partial z} &= -\epsilon_0 \frac{\partial E_x}{\partial t}
\end{align}$$

Si discretizamos el espacio y el tiempo con $z = k \Delta z$ y $t = n \Delta t$, usando la malla escalonada de Yee, podemos tomar el campo eléctrico como $E_x^{k,n+1/2}$ y el campo magnético como $H_y^{k+1/2,n+1}$.

Con esta notación, las ecuaciones en diferencias finitas quedan:

$$\begin{align}
\frac{E_x^{k,n+1/2} - E_x^{k,n-1/2}}{\Delta t} &= -\frac{1}{\epsilon_0} \frac{H_y^{k+1/2,n} - H_y^{k-1/2,n}}{\Delta z} \\
\frac{H_y^{k+1/2,n+1} - H_y^{k+1/2,n}}{\Delta t} &= -\frac{1}{\mu_0} \frac{E_x^{k+1,n+1/2} - E_x^{k,n+1/2}}{\Delta z}
\end{align}$$

Despejando $E_x^{k,n+1/2}$ y $H_y^{k+1/2,n+1}$, obtenemos las ecuaciones de actualización del método FDTD:

$$\begin{align}
E_x^{k,n+1/2} &= E_x^{k,n-1/2} - \frac{\Delta t}{\epsilon_0 \Delta z} \left[ H_y^{k+1/2,n} - H_y^{k-1/2,n} \right] \\
H_y^{k+1/2,n+1} &= H_y^{k+1/2,n} - \frac{\Delta t}{\mu_0 \Delta z} \left[ E_x^{k+1,n+1/2} - E_x^{k,n+1/2} \right]
\end{align}$$

### Formulación usando $B_y$ en lugar de $H_y$

Si usamos $\mathbf{B} = \mu_0 \mathbf{H}$, entonces las ecuaciones de Maxwell para esta geometría se escriben como:

$$\begin{align}
\frac{\partial E_x}{\partial z} &= -\frac{\partial B_y}{\partial t} \\
\frac{\partial B_y}{\partial z} &= -\mu_0 \epsilon_0 \frac{\partial E_x}{\partial t}
\end{align}$$

Al discretizar con la misma notación temporal y espacial, obtenemos:

$$\begin{align}
\frac{E_x^{k,n+1/2} - E_x^{k,n-1/2}}{\Delta t} &= -\frac{1}{\mu_0 \epsilon_0} \frac{B_y^{k+1/2,n} - B_y^{k-1/2,n}}{\Delta z} \\
\frac{B_y^{k+1/2,n+1} - B_y^{k+1/2,n}}{\Delta t} &= -\frac{E_x^{k+1,n+1/2} - E_x^{k,n+1/2}}{\Delta z}
\end{align}$$

y las ecuaciones de actualización quedan:

$$\begin{align}
E_x^{k,n+1/2} &= E_x^{k,n-1/2} - \frac{\Delta t}{\mu_0 \epsilon_0 \Delta z} \left[ B_y^{k+1/2,n} - B_y^{k-1/2,n} \right] \\
B_y^{k+1/2,n+1} &= B_y^{k+1/2,n} - \frac{\Delta t}{\Delta z} \left[ E_x^{k+1,n+1/2} - E_x^{k,n+1/2} \right]
\end{align}$$

Si además definimos $c = 1/\sqrt{\mu_0 \epsilon_0}$ y
$$
\beta = \frac{c}{\Delta z / \Delta t} = \frac{c \Delta t}{\Delta z},
$$
entonces las ecuaciones anteriores pueden escribirse como:

$$\begin{align}
E_x^{k,n+1/2} &= E_x^{k,n-1/2} - c \beta \left[ B_y^{k+1/2,n} - B_y^{k-1/2,n} \right] \\
B_y^{k+1/2,n+1} &= B_y^{k+1/2,n} - \frac{\beta}{c} \left[ E_x^{k+1,n+1/2} - E_x^{k,n+1/2} \right]
\end{align}$$

Recordar que a la hora de usar estas ecuaciones, el campo eléctrico se actualiza primero usando el campo magnético del paso de tiempo anterior, y luego el campo magnético se actualiza usando el nuevo campo eléctrico.

## Algoritmo FDTD
1. Inicializar los campos eléctricos y magnéticos en la malla.
2. Para cada paso de tiempo:
   a. Actualizar el campo eléctrico $E_x$ usando el campo magnético $B_y$ del paso de tiempo anterior.
   b. Actualizar el campo magnético $B_y$ usando el nuevo campo eléctrico $E_x$.
3. Aplicar condiciones de frontera según sea necesario (por ejemplo, condiciones de frontera absorbentes para simular un espacio infinito).
4. Graficar o almacenar los resultados según sea necesario para analizar la propagación de la onda electromagnética.
5. Continuar iterando hasta alcanzar el número deseado de pasos de tiempo o hasta que la onda haya propagado lo suficiente para el análisis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# Parámetros físicos
c = 1.0  # Velocidad de la luz
Nz = 300  # Número de puntos en la dirección z
dz = 1.0  # Paso espacial
beta = 0.4 # Condición de Courant para la estabilidad numérica
dt = beta * dz / c  # Paso de tiempo

steps = 600  # Número de pasos de tiempo

# Campos
E_x = np.zeros(Nz)  # Campo eléctrico
B_y = np.zeros(Nz)  # Campo magnético

# Pulso inicial gaussiano
z = np.arange(Nz)
E_x = np.exp(-((z - 80) / 15)**2)

# Figura
fig, ax = plt.subplots(figsize=(10, 5))
lineE, = ax.plot(z, E_x, label='E_x')
lineB, = ax.plot(z, B_y, label='B_y')
ax.set_xlim(0, Nz)
ax.set_ylim(-1.2, 1.2)
ax.set_xlabel('z')
ax.set_ylabel('Campo')
ax.legend()

# Actualización FDTD
def update(frame):
    global E_x, B_y
    
    # Actualizar campo eléctrico
    E_x[1:-1] += beta*(B_y[:-2] - B_y[1:-1])
    
    # Actualizar campo magnético
    B_y[1:-1] += -beta*(E_x[2:] - E_x[1:-1])

    #Condiciones de frontera
    E_x[0] = E_x[-1] = 0
    B_y[0] = B_y[-1] = 0
    
    lineE.set_ydata(E_x)
    lineB.set_ydata(B_y)
    return lineE, lineB

ani = FuncAnimation(
    fig, update, frames=steps, interval=20, blit=True
)

plt.show()